# Knowledge Distillation (RKD + Projector): ConvNeXt V2 → MobileNetV3 (Colab)

**Experiment 28 — Projector Head để tách gradient L_KD_cosine khỏi MagFace**

**Vấn đề Exp26 (task=2 kd=1 rkd_d=1 rkd_a=2):**
- `loss_kd` epoch40 = 0.9837 → student embedding **vuông góc** với teacher (cosine ≈ 0.016)
- Nguyên nhân: MagFace kéo `student_emb` về hướng `W_s` (student class centers),
  L_KD_cosine kéo về hướng `teacher_emb`. Hai gradient **xung đột** vì `W_s ≠ W_t`.

**Giải pháp Option 1 — Projector:**
```
student_emb (512-D)
      │
  ┌───┴────────────────┐
  │                    │
MagLinear          Projector (512→256→512)
(MagFace loss)          │
                   proj_emb (512-D)
                        │
               cosine_sim(proj_emb, teacher_emb)
                        │
                   L_KD_cosine
```

- Projector học **phép quay** từ student space → teacher space
- MagFace tối ưu `student_emb` theo class centers riêng của student
- Projector compensate rotation mismatch → L_KD_cosine có thể converge
- Inference: **không dùng projector** — chỉ export student backbone + embedding

| | Teacher | Student |
|---|---|---|
| Model | `MTLFaceRecognition` (ConvNeXt V2) | `FaceRecognitionMobileNetV3` |
| Params | ~28M | ~5M + Projector (~400K, training only) |
| Mode | **Frozen** | **Trainable** |

**Loss:**
```
L_total = 2·L_MagFace  +  1·L_KD_cosine(proj_emb, teacher_emb)  +  80·L_RKD_D  +  160·L_RKD_A
```

## 1. Mount Drive & Setup môi trường

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')
    print('Repo đã tồn tại, đã pull latest.')

%cd {REPO_DIR}
print(f'Working dir: {os.getcwd()}')

os.system('pip install -q albumentations==1.3.1 timm tabulate termcolor')

Mounted at /content/drive
/content/FR_Photometric_Stereo
Working dir: /content/FR_Photometric_Stereo


0

## 2. Imports & Cấu hình

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import albumentations as A
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.tensorboard import SummaryWriter
from tabulate import tabulate

from going_modular.dataloader.multitask import create_multitask_datafetcher, create_eval_loaders
from going_modular.model.MTLFaceRecognition import MTLFaceRecognition
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.loss.WeightClassMagLoss import WeightClassMagLoss
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.roc_auc_id import (
    compute_id_auc, compute_rank1,
    compute_id_auc_gallery_probe, compute_rank1_gallery_probe,
)
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint
from going_modular.utils.ExperimentManager import ExperimentManager

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

/content/FR_Photometric_Stereo
Device: cuda
GPU: NVIDIA L4


In [ ]:
# ════════════════════════════════════════════════════════════
#  CẤU HÌNH — Experiment: RKD + Projector Head
# ════════════════════════════════════════════════════════════

DRIVE_DATASET_DIR = '/content/drive/MyDrive/Photometric_DB_Full/'

TEACHER_CKPT = '/content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'

EXPERIMENT_NAME = '(new_angl-1-1-100-200)KD_RKD_ConvNextV2_to_MobileNetV3_Albedo_projector'

CONFIGURATION = {
    'note':        EXPERIMENT_NAME,
    'dataset_dir': DRIVE_DATASET_DIR,
    'output_dir':  '/content/drive/MyDrive/',

    'type':        'albedo',

    'teacher_backbone': 'convnextv2_tiny',
    'backbone':         'mobilenetv3_large_100',

    'use_sampler': True,
    'device':      device,
    'epochs':      100,
    'batch_size':  32,
    'image_size':  112,
    'base_lr':     1e-4,
    'num_classes': None,

    # ── So sánh với các experiments trước ────────────────────────────────
    # Exp26: task=2 kd=1 rkd_d=1  rkd_a=2   → rank-1=78%, loss_kd không converge
    # Exp27: task=2 kd=0 rkd_d=80 rkd_a=160 → (đang chạy)
    # Exp28: task=2 kd=1 rkd_d=80 rkd_a=160 → dùng projector, kd có thể converge
    #
    # Projector (512→256→512) học phép quay student space → teacher space.
    # MagFace tối ưu student_emb theo class centers riêng, không bị nhiễu bởi L_kd.
    # RKD-D/A weights đặt bằng Exp27 để so sánh công bằng.
    'task_weight':  1.0,
    'kd_weight':    1.0,    # L_KD_cosine(projector(student_emb), teacher_emb)
    'rkd_d_weight': 100.0,
    'rkd_a_weight': 200.0,

    # Projector architecture
    'proj_in_dim':     512,
    'proj_hidden_dim': 512,
    'proj_out_dim':    512,
}

print(f"Dataset dir : {CONFIGURATION['dataset_dir']}")
print(f"Output dir  : {CONFIGURATION['output_dir']}")
print(f"Teacher ckpt: {TEACHER_CKPT}")
print(f"Loss weights: task={CONFIGURATION['task_weight']}  "
      f"kd={CONFIGURATION['kd_weight']}  "
      f"rkd_d={CONFIGURATION['rkd_d_weight']}  "
      f"rkd_a={CONFIGURATION['rkd_a_weight']}")
print(f"Projector   : {CONFIGURATION['proj_in_dim']}→{CONFIGURATION['proj_hidden_dim']}→{CONFIGURATION['proj_out_dim']}")

Dataset dir : /content/drive/MyDrive/Photometric_DB_Full/
Output dir  : /content/drive/MyDrive/
Teacher ckpt: /content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth
Loss weights: task=1.0  kd=1.0  rkd_d=80.0  rkd_a=160.0
Projector   : 512→512→512


## 3. Data Loading

In [ ]:
dataset_dir = CONFIGURATION['dataset_dir']

train_csv = os.path.join(dataset_dir, 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'dataset', 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'train_set.csv')
if not os.path.exists(train_csv):
    raise FileNotFoundError(
        f'Không tìm thấy CSV train tại {dataset_dir}.\n'
        f'Kiểm tra lại DRIVE_DATASET_DIR.'
    )
print(f'Train CSV: {train_csv}')

df_train = pd.read_csv(train_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].nunique())
print(f'num_classes : {CONFIGURATION["num_classes"]}')
print(f'Số mẫu train: {len(df_train)}')

train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
    A.HorizontalFlip(p=0.5),
])
test_transform = A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size']),
])

train_dl, test_dl, _ = create_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform, 'train_split.csv', 'probe_split.csv'
)
print(f'Train batches: {len(train_dl)} | Test batches (probe): {len(test_dl)}')

gallery_dl, probe_dl = create_eval_loaders(CONFIGURATION, test_transform)

Train CSV: /content/drive/MyDrive/Photometric_DB_Full/train_split.csv
num_classes : 352
Số mẫu train: 2622
>>> SingleLoader: MODE = PK SAMPLER
Train batches: 81 | Test batches (probe): 9
Gallery: 68 ảnh | Probe: 288 ảnh
Shared identity space: 68 identities


## 4. Teacher Model (ConvNeXt V2 — Frozen)

In [ ]:
if not os.path.exists(TEACHER_CKPT):
    raise FileNotFoundError(
        f'Không tìm thấy teacher checkpoint: {TEACHER_CKPT}\n'
        f'Kiểm tra lại biến TEACHER_CKPT.'
    )

teacher = MTLFaceRecognition(
    backbone=CONFIGURATION['teacher_backbone'],
    num_classes=CONFIGURATION['num_classes'],
)

# Load weights
ckpt = torch.load(TEACHER_CKPT, map_location=device, weights_only=False)
state_dict = ckpt['model_state_dict']

# Remove keys related to the id_head.maglinear layer to avoid size mismatch errors
# This layer is not needed as the teacher is used for feature extraction only
keys_to_remove = [k for k in state_dict.keys() if 'id_head.maglinear' in k]
for k in keys_to_remove:
    del state_dict[k]

teacher.load_state_dict(ckpt['model_state_dict'], strict=False)  # strict=False để bỏ qua nếu có layers mới (như classifier)
print(f"Teacher loaded — epoch {ckpt.get('epoch', '?')}")

teacher.to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

teacher_params = sum(p.numel() for p in teacher.parameters())
print(f'Teacher params: {teacher_params:,} (tất cả frozen)')

with torch.no_grad():
    _dummy = torch.randn(2, 3, 112, 112).to(device)
    _t_emb = teacher.get_embedding(_dummy)[-1]
    print(f'Teacher ID embedding shape: {_t_emb.shape}')

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

Teacher loaded — epoch 28
Teacher params: 45,670,280 (tất cả frozen)
Teacher ID embedding shape: torch.Size([2, 512])


## 4.1. Đánh giá Teacher Model (baseline)

In [ ]:
class _TeacherEvalWrapper(torch.nn.Module):
    def __init__(self, teacher):
        super().__init__()
        self._teacher = teacher

    def get_embedding(self, x):
        return self._teacher.get_embedding(x)[-1]


teacher_wrapper = _TeacherEvalWrapper(teacher).to(device)
teacher_wrapper.eval()

teacher_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, teacher_wrapper, device)
teacher_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, teacher_wrapper, device)

rows = [
    ['Cosine AUC    (gallery→probe)', f"{teacher_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)', f"{teacher_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)', f"{teacher_gp_rank1:.4f}"],
]
print(f"Teacher ({CONFIGURATION['teacher_backbone']})")
print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='fancy_grid'))

Teacher (convnextv2_tiny)
╒═══════════════════════════════╤═════════╕
│ Metric                        │   Value │
╞═══════════════════════════════╪═════════╡
│ Cosine AUC    (gallery→probe) │  0.9761 │
├───────────────────────────────┼─────────┤
│ Euclidean AUC (gallery→probe) │  0.9761 │
├───────────────────────────────┼─────────┤
│ Rank-1 Acc    (gallery→probe) │  0.8021 │
╘═══════════════════════════════╧═════════╛


## 5. Student Model + Projector Head

In [ ]:
student = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
student.to(device)

total_p     = sum(p.numel() for p in student.parameters())
trainable_p = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Student total params    : {total_p:,}')
print(f'Student trainable params: {trainable_p:,}')

with torch.no_grad():
    _s_emb = student.get_embedding(_dummy)
    print(f'Student embedding shape: {_s_emb.shape}')

model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

Student total params    : 3,645,744
Student trainable params: 3,645,744
Student embedding shape: torch.Size([2, 512])


In [ ]:
# ── Kiến trúc Student ────────────────────────────────────────────────────────
print('=' * 60)
print('STUDENT MODEL ARCHITECTURE')
print('=' * 60)
print(student)
print()

# ── Thống kê params từng component ──────────────────────────────────────────
def count_params(module):
    total     = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

rows = []
for name, module in [('backbone',  student.backbone),
                     ('embedding', student.embedding),
                     ('maglinear', student.maglinear)]:
    total, trainable = count_params(module)
    rows.append([name, f'{total:,}', f'{trainable:,}'])

total_all,     trainable_all     = count_params(student)
rows.append(['─' * 10, '─' * 12, '─' * 12])
rows.append(['TOTAL', f'{total_all:,}', f'{trainable_all:,}'])

print(tabulate(rows, headers=['Component', 'Params', 'Trainable'], tablefmt='fancy_grid'))

STUDENT MODEL ARCHITECTURE
FaceRecognitionMobileNetV3(
  (backbone): MIMobileNetV3(
    (backbone): MobileNetV3Features(
      (conv_stem): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act1): Hardswish()
      (blocks): Sequential(
        (0): Sequential(
          (0): DepthwiseSeparableConv(
            (conv_dw): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (bn1): BatchNormAct2d(
              16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
              (drop): Identity()
              (act): ReLU(inplace=True)
            )
            (aa): Identity()
            (se): Identity()
            (conv_pw): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn2): BatchNormAct2d(
              16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=Tru

In [ ]:
class Projector(nn.Module):
    """
    Ánh xạ student embedding space → teacher embedding space.

    Mục đích: hấp thụ sự khác biệt về hướng (rotation mismatch) giữa hai không gian
    embedding, cho phép L_KD_cosine converge mà không xung đột với MagFace.

    Chỉ dùng trong training. KHÔNG export khi deploy.
    Architecture: Linear → BN → ReLU → Linear
    """

    def __init__(self, in_dim: int = 512, hidden_dim: int = 256, out_dim: int = 512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


projector = Projector(
    in_dim=CONFIGURATION['proj_in_dim'],
    hidden_dim=CONFIGURATION['proj_hidden_dim'],
    out_dim=CONFIGURATION['proj_out_dim'],
).to(device)

proj_params = sum(p.numel() for p in projector.parameters())
print(f'Projector params: {proj_params:,} (training only, không export)')

# Kiểm tra forward pass
with torch.no_grad():
    _proj_out = projector(_s_emb)
    print(f'Projector output shape: {_proj_out.shape}')

Projector params: 525,312 (training only, không export)
Projector output shape: torch.Size([2, 512])


## 6. RKD Loss với Projector

```
L_total = α·L_MagFace(student_emb)  +  β·L_KD_cosine(proj(student_emb), teacher_emb)  +  γ·L_RKD_D  +  δ·L_RKD_A
```

**Luồng gradient:**
```
L_MagFace  →  student_emb  →  backbone          (kéo emb về class centers của student)
L_KD       →  proj_emb  →  projector  →  student_emb  →  backbone
                             (projector hấp thụ rotation, backbone nhận tín hiệu KD yếu hơn)
L_RKD_D/A  →  student_emb  →  backbone          (bảo toàn relational structure)
```

So với Exp26 (không có projector): L_KD gradient trực tiếp xung đột với L_MagFace.
Với projector: projector có thể học `proj(W_s_c) ≈ W_t_c` (map student class centers về teacher class centers) → xung đột giảm đáng kể.

In [ ]:
class RKDLossWithProjector(nn.Module):
    """
    L_total = task_w*L_MagFace + kd_w*L_KD_cosine(proj_emb, teacher_emb)
            + rkd_d_w*L_RKD_D(student_emb, teacher_emb)
            + rkd_a_w*L_RKD_A(student_emb, teacher_emb)

    L_KD_cosine dùng proj_emb = projector(student_emb) thay vì student_emb trực tiếp.
    RKD-D/A vẫn dùng student_emb gốc để bảo toàn relational structure thực sự.
    """

    def __init__(
        self,
        metadata_path: str,
        task_weight:   float = 2.0,
        kd_weight:     float = 1.0,
        rkd_d_weight:  float = 80.0,
        rkd_a_weight:  float = 160.0,
    ):
        super().__init__()
        self.magface  = WeightClassMagLoss(metadata_path)
        self.task_w   = task_weight
        self.kd_w     = kd_weight
        self.rkd_d_w  = rkd_d_weight
        self.rkd_a_w  = rkd_a_weight

    @staticmethod
    def _pdist(e: torch.Tensor) -> torch.Tensor:
        diff = e.unsqueeze(0) - e.unsqueeze(1)
        return diff.pow(2).sum(-1).clamp(min=1e-12).sqrt()

    def _rkd_distance(self, s_emb: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        B = s_emb.size(0)
        mask = ~torch.eye(B, dtype=torch.bool, device=s_emb.device)

        with torch.no_grad():
            td   = self._pdist(t_emb)
            mu_t = td[mask].mean()
            td_n = td / (mu_t + 1e-8)

        sd   = self._pdist(s_emb)
        mu_s = sd[mask].mean()
        sd_n = sd / (mu_s + 1e-8)

        return F.huber_loss(sd_n[mask], td_n[mask], delta=1.0, reduction='mean')

    def _rkd_angle(self, s_emb: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        B       = s_emb.size(0)
        eye     = torch.eye(B, dtype=torch.bool, device=s_emb.device)
        mask_3d = (~eye.unsqueeze(2)) & (~eye.unsqueeze(1))

        def _angle_matrix(e: torch.Tensor) -> torch.Tensor:
            diff = e.unsqueeze(0) - e.unsqueeze(1)
            norm = diff.norm(p=2, dim=2, keepdim=True).clamp(min=1e-8)
            diff = diff / norm
            return torch.bmm(diff, diff.transpose(1, 2))

        with torch.no_grad():
            ta = _angle_matrix(t_emb)
        sa = _angle_matrix(s_emb)

        return F.huber_loss(sa[mask_3d], ta[mask_3d], delta=1.0, reduction='mean')

    def forward(
        self,
        student_logits,
        student_norm,
        student_emb,   # [B, 512] — dùng cho RKD và MagFace
        proj_emb,      # [B, 512] — projector(student_emb), dùng cho L_KD_cosine
        teacher_emb,   # [B, 512]
        id_labels,
    ):
        l_task = self.magface(student_logits, id_labels, student_norm)

        # L_KD_cosine: dùng proj_emb thay vì student_emb trực tiếp
        s_proj_n = F.normalize(proj_emb,    p=2, dim=1)
        t_n      = F.normalize(teacher_emb, p=2, dim=1)
        l_kd     = (1.0 - F.cosine_similarity(s_proj_n, t_n, dim=1)).mean()

        # RKD: dùng student_emb gốc để bảo toàn relational structure thực sự
        l_rkd_d = self._rkd_distance(student_emb, teacher_emb)
        l_rkd_a = self._rkd_angle(student_emb, teacher_emb)

        total = (
            self.task_w  * l_task
          + self.kd_w    * l_kd
          + self.rkd_d_w * l_rkd_d
          + self.rkd_a_w * l_rkd_a
        )
        return total, l_task, l_kd, l_rkd_d, l_rkd_a


criterion = RKDLossWithProjector(
    metadata_path=train_csv,
    task_weight=CONFIGURATION['task_weight'],
    kd_weight=CONFIGURATION['kd_weight'],
    rkd_d_weight=CONFIGURATION['rkd_d_weight'],
    rkd_a_weight=CONFIGURATION['rkd_a_weight'],
)
print('RKDLossWithProjector khởi tạo thành công.')
print(f"  task={CONFIGURATION['task_weight']}  "
      f"kd={CONFIGURATION['kd_weight']} (qua projector)  "
      f"rkd_d={CONFIGURATION['rkd_d_weight']}  "
      f"rkd_a={CONFIGURATION['rkd_a_weight']}")

RKDLossWithProjector khởi tạo thành công.
  task=1.0  kd=1.0 (qua projector)  rkd_d=80.0  rkd_a=160.0


## 7. Training

In [ ]:
def train_epoch(train_dl, teacher, student, projector, criterion, optimizer, device):
    student.train()
    projector.train()

    total_loss = total_task = total_kd = total_rkd_d = total_rkd_a = 0.0

    for X, y in train_dl:
        X, y      = X.to(device), y.to(device)
        id_labels = y[:, 0]

        with torch.no_grad():
            teacher_emb = teacher.get_embedding(X)[-1]   # [B, 512], frozen

        feat        = student.backbone(X)                # [B, 512, H, W]
        student_emb = student.embedding(feat)            # [B, 512]
        logits, norm = student.maglinear(student_emb)

        # Projector maps student_emb → teacher embedding space
        # Gradient từ L_KD_cosine chạy: l_kd → proj_emb → projector → student_emb → backbone
        proj_emb = projector(student_emb)

        loss, l_task, l_kd, l_rkd_d, l_rkd_a = criterion(
            logits, norm, student_emb, proj_emb, teacher_emb, id_labels
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss  += loss.item()
        total_task  += l_task.item()
        total_kd    += l_kd.item()
        total_rkd_d += l_rkd_d.item()
        total_rkd_a += l_rkd_a.item()

    n = len(train_dl)
    return (
        total_loss  / n,
        total_task  / n,
        total_kd    / n,
        total_rkd_d / n,
        total_rkd_a / n,
    )


def display_metrics(epoch, train_metrics, test_metrics):
    rows = []
    for k in train_metrics:
        tv = train_metrics[k]
        ev = test_metrics.get(k, '-')
        fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
        rows.append([k, fmt(tv), fmt(ev)])
    print(f'\nEp {epoch}:')
    print(tabulate(rows, headers=['Metric', 'Train', 'Test'], tablefmt='fancy_grid'))

In [ ]:
# Optimizer chung cho student + projector
# Projector và student cùng learning rate để chúng học song song
optimizer = Adam(
    list(student.parameters()) + list(projector.parameters()),
    lr=CONFIGURATION['base_lr'],
)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

manager = ExperimentManager(CONFIGURATION)
manager.log_text(
    f"Teacher: {CONFIGURATION['teacher_backbone']} | "
    f"Student: {CONFIGURATION['backbone']} | "
    f"Modality: {CONFIGURATION['type']} | "
    f"task={CONFIGURATION['task_weight']} kd={CONFIGURATION['kd_weight']} (projector) "
    f"rkd_d={CONFIGURATION['rkd_d_weight']} rkd_a={CONFIGURATION['rkd_a_weight']} | "
    f"Projector: {CONFIGURATION['proj_in_dim']}→{CONFIGURATION['proj_hidden_dim']}→{CONFIGURATION['proj_out_dim']}"
)

ckpt_saver = ModelCheckpoint(
    output_dir=manager.ckpt_dir,
    mode='max',
    best_metric_name='auc_id_cosine',
)
early_stopping = MultiMetricEarlyStopping(
    monitor_keys=['auc_id_cosine'],
    patience=10,
    mode='max',
    verbose=1,
    save_dir=manager.ckpt_dir,
    start_from_epoch=5,
)

writer = SummaryWriter(log_dir=manager.log_dir)
print(f'Experiment dir: {manager.exp_dir}')
print(f'Checkpoint dir: {manager.ckpt_dir}')

KHOI TAO THI NGHIEM: (new_angl-1-1-80-160)KD_RKD_ConvNextV2_to_MobileNetV3_Albedo_projector
Luu tru tai: /content/drive/MyDrive/experiments/(new_angl-1-1-80-160)KD_RKD_ConvNextV2_to_MobileNetV3_Albedo_projector
Thoi gian: 2026-06-09 13:23:28
--------------------------------------------------
Teacher: convnextv2_tiny | Student: mobilenetv3_large_100 | Modality: albedo | task=1.0 kd=1.0 (projector) rkd_d=80.0 rkd_a=160.0 | Projector: 512→512→512
Experiment dir: /content/drive/MyDrive/experiments/(new_angl-1-1-80-160)KD_RKD_ConvNextV2_to_MobileNetV3_Albedo_projector
Checkpoint dir: /content/drive/MyDrive/experiments/(new_angl-1-1-80-160)KD_RKD_ConvNextV2_to_MobileNetV3_Albedo_projector/checkpoints


In [ ]:
START_EPOCH = 0

manager.log_text('BAT DAU RKD + PROJECTOR KNOWLEDGE DISTILLATION')

for epoch in range(START_EPOCH, CONFIGURATION['epochs']):
    manager.log_text(f'\n--- Epoch {epoch+1}/{CONFIGURATION["epochs"]} ---')

    train_loss, train_task, train_kd, train_rkd_d, train_rkd_a = train_epoch(
        train_dl, teacher, student, projector, criterion, optimizer, device
    )

    train_auc = compute_id_auc(train_dl, student, device)
    test_auc  = compute_id_auc(test_dl,  student, device)

    train_metrics = {
        'loss':             train_loss,
        'loss_task':        train_task,
        'loss_kd':          train_kd,
        'loss_rkd_d':       train_rkd_d,
        'loss_rkd_a':       train_rkd_a,
        'auc_id_cosine':    train_auc['id_cosine'],
        'auc_id_euclidean': train_auc['id_euclidean'],
    }
    test_metrics = {
        'auc_id_cosine':    test_auc['id_cosine'],
        'auc_id_euclidean': test_auc['id_euclidean'],
    }

    writer.add_scalar('Loss/total',   train_loss,   epoch + 1)
    writer.add_scalar('Loss/task',    train_task,   epoch + 1)
    writer.add_scalar('Loss/kd',      train_kd,     epoch + 1)
    writer.add_scalar('Loss/rkd_d',   train_rkd_d,  epoch + 1)
    writer.add_scalar('Loss/rkd_a',   train_rkd_a,  epoch + 1)
    writer.add_scalars('AUC/cosine',
        {'train': train_auc['id_cosine'],    'test': test_auc['id_cosine']},    epoch + 1)
    writer.add_scalars('AUC/euclidean',
        {'train': train_auc['id_euclidean'], 'test': test_auc['id_euclidean']}, epoch + 1)

    display_metrics(epoch + 1, train_metrics, test_metrics)
    manager.log_metrics(epoch + 1, {**train_metrics, **test_metrics})

    ckpt_saver(student, optimizer, epoch + 1, test_metrics, scheduler)
    early_stopping(test_metrics, student, epoch + 1)
    scheduler.step(epoch)

    if early_stopping.early_stop:
        manager.log_text('Early stopping triggered.')
        break

writer.close()
manager.log_text('RKD + PROJECTOR KD HOAN TAT.')

BAT DAU RKD + PROJECTOR KNOWLEDGE DISTILLATION

--- Epoch 1/100 ---

Ep 1:
╒══════════════════╤═════════╤════════╕
│ Metric           │   Train │ Test   │
╞══════════════════╪═════════╪════════╡
│ loss             │ 31.1492 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_task        │ 28.9845 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_kd          │  0.9497 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_rkd_d       │  0.0054 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_rkd_a       │  0.0049 │ -      │
├──────────────────┼─────────┼────────┤
│ auc_id_cosine    │  0.9176 │ 0.8860 │
├──────────────────┼─────────┼────────┤
│ auc_id_euclidean │  0.9176 │ 0.8860 │
╘══════════════════╧═════════╧════════╛
Ep 1: loss: 31.1492, loss_task: 28.9845, loss_kd: 0.9497, loss_rkd_d: 0.0054, loss_rkd_a: 0.0049, auc_id_cosine: 0.8860, auc_id_euclidean: 0.8860
--> SAVE BEST MODEL (auc_id_cosine: 0.8860)

--- Epoch 2/100 ---

Ep 2:
╒══════════════════╤═══════

## 8. Resume Training từ Checkpoint

> Checkpoint lưu cả `student` lẫn `projector` state.  
> **Cách dùng:** Chạy cell Setup → Imports → Data → Teacher → Student → Projector → Loss → Setup Train,  
> sau đó chạy cell này, rồi chạy lại cell fit.

In [ ]:
# CKPT_PATH = os.path.join(manager.ckpt_dir, 'last_model.pth')

# if not os.path.exists(CKPT_PATH):
#     raise FileNotFoundError(f'Không tìm thấy checkpoint: {CKPT_PATH}')

# checkpoint = torch.load(CKPT_PATH, map_location=device)
# student.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# if 'scheduler_state_dict' in checkpoint:
#     scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
# if 'projector_state_dict' in checkpoint:
#     projector.load_state_dict(checkpoint['projector_state_dict'])
#     print('Projector state restored.')
# else:
#     print('WARNING: projector_state_dict không có trong checkpoint — projector re-init từ đầu.')

# START_EPOCH = checkpoint['epoch']
# print(f'Resume từ epoch {START_EPOCH} — chạy lại cell "cell-fit" để tiếp tục.')

## 8.1. Lưu projector state vào checkpoint thủ công

> Nếu `ModelCheckpoint` của bạn chưa hỗ trợ lưu projector, chạy cell này sau mỗi epoch quan trọng.

In [ ]:
# Lưu cả student + projector vào một file riêng
combined_ckpt_path = os.path.join(manager.ckpt_dir, 'last_model_with_projector.pth')
torch.save({
    'model_state_dict':     student.state_dict(),
    'projector_state_dict': projector.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'epoch':                START_EPOCH,
}, combined_ckpt_path)
print(f'Saved combined checkpoint: {combined_ckpt_path}')

Saved combined checkpoint: /content/drive/MyDrive/experiments/(new_angl-1-1-80-160)KD_RKD_ConvNextV2_to_MobileNetV3_Albedo_projector/checkpoints/last_model_with_projector.pth


## 9. Đánh giá Final

> Inference **không dùng projector** — chỉ dùng student backbone + embedding layer trực tiếp.

In [ ]:
best_ckpt_path = os.path.join(manager.ckpt_dir, 'best_model.pth')
best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
student.load_state_dict(best_ckpt['model_state_dict'])
student.eval()
print(f"Best model từ epoch {best_ckpt.get('epoch', '?')}")
print('(Inference không dùng projector — student.get_embedding() trực tiếp)')

student_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, student, device)
student_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, student, device)

rows = [
    ['Cosine AUC    (gallery→probe)', f"{student_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)', f"{student_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)', f"{student_gp_rank1:.4f}"],
]
print(f"\nStudent ({CONFIGURATION['backbone']}) — RKD+Projector from {CONFIGURATION['teacher_backbone']}")
print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='fancy_grid'))

Best model từ epoch 51
(Inference không dùng projector — student.get_embedding() trực tiếp)

Student (mobilenetv3_large_100) — RKD+Projector from convnextv2_tiny
╒═══════════════════════════════╤═════════╕
│ Metric                        │   Value │
╞═══════════════════════════════╪═════════╡
│ Cosine AUC    (gallery→probe) │  0.9708 │
├───────────────────────────────┼─────────┤
│ Euclidean AUC (gallery→probe) │  0.9708 │
├───────────────────────────────┼─────────┤
│ Rank-1 Acc    (gallery→probe) │  0.7743 │
╘═══════════════════════════════╧═════════╛


In [ ]:
best_ckpt_path = os.path.join(manager.ckpt_dir, 'last_model.pth')
best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
student.load_state_dict(best_ckpt['model_state_dict'])
student.eval()
print(f"last model từ epoch {best_ckpt.get('epoch', '?')}")
print('(Inference không dùng projector — student.get_embedding() trực tiếp)')

student_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, student, device)
student_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, student, device)

rows = [
    ['Cosine AUC    (gallery→probe)', f"{student_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)', f"{student_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)', f"{student_gp_rank1:.4f}"],
]
print(f"\nStudent ({CONFIGURATION['backbone']}) — RKD+Projector from {CONFIGURATION['teacher_backbone']}")
print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='fancy_grid'))

last model từ epoch 61
(Inference không dùng projector — student.get_embedding() trực tiếp)

Student (mobilenetv3_large_100) — RKD+Projector from convnextv2_tiny
╒═══════════════════════════════╤═════════╕
│ Metric                        │   Value │
╞═══════════════════════════════╪═════════╡
│ Cosine AUC    (gallery→probe) │  0.9704 │
├───────────────────────────────┼─────────┤
│ Euclidean AUC (gallery→probe) │  0.9704 │
├───────────────────────────────┼─────────┤
│ Rank-1 Acc    (gallery→probe) │  0.7778 │
╘═══════════════════════════════╧═════════╛


## 9.1. So sánh Teacher vs Exp26 vs Exp27 vs Exp28

In [ ]:
# # Cập nhật kết quả Exp26 và Exp27 khi có
# exp26_rank1      = 0.78    # task=2 kd=1 rkd_d=1  rkd_a=2
# exp27_rank1      = None    # task=2 kd=0 rkd_d=80 rkd_a=160 (đang chạy)

# fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else ('running' if v is None else str(v))

# compare_rows = [
#     ['Config',
#      'MTL+FSM+GRL (teacher)',
#      'task=2 kd=1 rkd_d=1 rkd_a=2',
#      'task=2 kd=0 rkd_d=80 rkd_a=160',
#      'task=2 kd=1(proj) rkd_d=80 rkd_a=160'],
#     ['Cosine AUC (gal→prb)',
#      f"{teacher_gp_auc['id_cosine']:.4f}",
#      '-', '-',
#      f"{student_gp_auc['id_cosine']:.4f}"],
#     ['Rank-1 (gal→prb)',
#      f"{teacher_gp_rank1:.4f}",
#      fmt(exp26_rank1),
#      fmt(exp27_rank1),
#      f"{student_gp_rank1:.4f}"],
# ]
# print(tabulate(
#     compare_rows,
#     headers=['Metric', 'Teacher', 'Exp26', 'Exp27 (no KD)', 'Exp28 (projector)'],
#     tablefmt='fancy_grid'
# ))

## 10. Export ONNX

Chỉ export student backbone + embedding (không có projector, không có MagLinear).

In [ ]:
# class InferenceWrapper(nn.Module):
#     """Backbone + embedding + L2 normalize. Không có MagLinear, không có Projector."""
#     def __init__(self, model):
#         super().__init__()
#         self.backbone  = model.backbone
#         self.embedding = model.embedding

#     def forward(self, x):
#         emb = self.embedding(self.backbone(x))
#         return F.normalize(emb, p=2, dim=1)


# inference_model = InferenceWrapper(student).eval().cpu()
# dummy_input = torch.randn(1, 3, 112, 112)

# onnx_path = os.path.join(manager.ckpt_dir, 'rkd_projector_mobilenetv3_fr.onnx')

# torch.onnx.export(
#     inference_model,
#     dummy_input,
#     onnx_path,
#     input_names=['input'],
#     output_names=['embedding'],
#     dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
#     opset_version=17,
# )
# print(f'Exported ONNX (no projector): {onnx_path}')